In [2]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.profiler import profile, ProfilerActivity, schedule, record_function

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

def collate_fn(
    tokenizer: AutoTokenizer, batch: list[str]
) -> tuple[torch.Tensor,  torch.Tensor]:
    encoded_batch = tokenizer.batch_encode_plus(
        batch, padding="longest", return_tensors="pt", return_token_type_ids=False)
    return encoded_batch.to(device)


def get_loaders(path = "blo05/cleaned_wiki_en_20-40", from_csv=False, batch_size=32):
    if from_csv:
        ds = load_dataset("csv", data_files=path)['train'].filter(lambda x: len(x['text']) <=150 )
    else:
        ds = load_dataset(path)['train'].filter(lambda x: len(x['text']) <=150)


    ds_split = ds.train_test_split(test_size=0.15, seed=777)

    ds_train =  ds_split['train']['text']
    ds_val =  ds_split['test']['text']
    print(len(ds_train),len(ds_val))
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=lambda batch:collate_fn(tokenizer,batch))
    val_loader = DataLoader(ds_val, batch_size=batch_size, collate_fn=lambda batch:collate_fn(tokenizer,batch))

    return train_loader, val_loader




In [4]:
BATCH_SIZE = 64
train_loader, val_loader = get_loaders(batch_size=BATCH_SIZE)

len(train_loader), len(val_loader)


cleaned_wiki_en_20-40.csv:   0%|          | 0.00/1.99G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1215497 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1215497 [00:00<?, ? examples/s]

20215 3568


(316, 56)

In [5]:
from dataclasses import dataclass
import math
from torch import nn
from torch.nn import Linear

import torch.nn.functional as F

@dataclass
class LlamaConfig():
    n_layers: int
    n_heads: int
    vocab_size: int
    hidden_size: int
    max_seq_len: int = 2000  # Нужно для RoPe, чтобы не создавть RoPe при новом проходе заново, а брать углы поворота до seq_len.


class Llama(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()

        self.config = config
        self.n_layers = config.n_layers

        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.llama_layers = nn.ModuleList([LlamaLayer(config) for i in range(self.n_layers)])
        self.logits_layer = LogitsLayer(config)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.llama_layers:
            x = layer(x)

        return self.logits_layer(x)


class LogitsLayer(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config
        self.W = Linear(config.hidden_size, config.vocab_size)

    def forward(self, x):
        return self.W(x)


class LlamaLayer(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config
        self.self_attention = SelfAttention(config)
        self.ffn = FFN(config)
        self.rms_norm = nn.RMSNorm(config.hidden_size)

    def forward(self, x):
        # x.shape = (B, seq_len, hidden_dim
        x_normed = self.rms_norm(x)
        x = x + self.self_attention(x_normed)

        x_normed = self.rms_norm(x)
        x = x + self.ffn(x)

        return x


class FFN(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config
        self.W1 = Linear(config.hidden_size, config.hidden_size)
        self.W2 = Linear(config.hidden_size, config.hidden_size)
        self.W3 = Linear(config.hidden_size, config.hidden_size)

    def forward(self, x):
        return self.W3(F.silu(self.W1(x)) * self.W2(x))


class SelfAttention(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config
        self.n_heads = config.n_heads
        self.head_dim = config.hidden_size // config.n_heads
        self.WQ = Linear(config.hidden_size, config.hidden_size)
        self.WK = Linear(config.hidden_size, config.hidden_size)
        self.WV = Linear(config.hidden_size, config.hidden_size)
        self.WO = Linear(config.hidden_size, config.hidden_size)
        self.RoPe = RoPE(config)

    def forward(self, x):
        B, seq_len, hidden_size = x.shape

        q = self.WQ(x)
        k = self.WK(x)
        v = self.WV(x)
        q = q.view(B, seq_len, self.n_heads, self.head_dim)
        k = k.view(B, seq_len, self.n_heads, self.head_dim)
        v = v.view(B, seq_len, self.n_heads, self.head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        q, k = self.RoPe(q, k)

        attn = nn.functional.softmax(
            torch.where(
                torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).to(torch.bool), #Главную диагональ не маскируем
                -torch.inf,
                (q @ k.transpose(-2, -1)) * (1 / math.sqrt(self.head_dim)),
            ),
            dim=-1,
        )

        out = self.WO((attn @ v).view(B, seq_len, hidden_size))
        return out

class RoPE(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config
        self.max_seq_len = config.max_seq_len
        self.base = 10000.0
        self.head_dim = config.hidden_size // config.n_heads
        self.t = self.base ** (- 2 * (torch.arange(0, self.head_dim // 2, dtype=torch.float32, device=device) - 1) / (self.head_dim))
        self.range = torch.arange(0, self.max_seq_len, dtype=torch.float32, device=device)

        self.temp = self.range.view(-1, 1) * self.t.view(1, -1)

        self.cos = torch.cos(self.temp)
        self.sin = torch.sin(self.temp)

    def split(self, x):
        # x.shape = (B, n_head, seq_len, head_dim)

        first_half = x[..., :self.head_dim // 2]
        second_half = x[..., self.head_dim // 2:]
        return first_half, second_half

    def rotate(self, x):
        # x.shape = (B, n_head, seq_len, head_dim)

        first_half, second_half = self.split(x)
        seq_len = x.size(-2)
        return torch.cat(
            [
                first_half * self.sin[:seq_len] + second_half * self.cos[:seq_len],
                first_half * self.cos[:seq_len] - second_half * self.sin[:seq_len]
            ],
            dim=-1
        )

    def forward(self, q, k):
        # (q|k).shape = (B, n_head, seq_len, head_dim)

        return self.rotate(q), self.rotate(k)

class RMSNorm(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config
        self.scale = nn.Parameter(torch.ones(config.hidden_size))

    def forward(self, x):
        return (x * torch.rsqrt((x**2).mean(dim = -1, keepdim=True))) * self.scale



In [10]:
def paramets_amount(model):
  return sum(p.numel() for p in model.parameters())

In [52]:
deep_model = Llama(LlamaConfig(n_layers=20, n_heads=4, vocab_size=len(tokenizer), hidden_size=72)).to(device)
wide_model = Llama(LlamaConfig(n_layers=1, n_heads=4, vocab_size=len(tokenizer), hidden_size=88)).to(device)

paramets_amount(deep_model), paramets_amount(wide_model)

(5162970, 5457306)

In [53]:
def on_trace_ready(prof):
    prof.export_chrome_trace("trace.json")

In [62]:
opt = torch.optim.AdamW(deep_model.parameters())
loss_fun = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [63]:
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=schedule(
        skip_first=1,
        wait=1,
        warmup=1,
        active=3,
        repeat=2,
    ),
    record_shapes=True,
    on_trace_ready=on_trace_ready,
) as prof:
    for _, batch in zip(trange(14), train_loader):
        input = batch['input_ids'].to(device)

        logits = deep_model(input)[:,:-1,:].contiguous()
        targets = input[:,1:].to(device).contiguous()

        loss = loss_fun(logits.reshape(-1, len(tokenizer)), targets.reshape(-1))

        opt.zero_grad()
        loss.backward()
        opt.step()

        prof.step()

  0%|          | 0/14 [00:00<?, ?it/s]

In [56]:
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=10
))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     209.792ms        80.96%     209.792ms      69.931ms             3  
                                          ProfilerStep*        23.94%     371.052ms        85.24%        1.321s     440.446ms       0.000us         0.00%      87.302ms      29.101ms             3  
    autog

In [64]:
opt = torch.optim.AdamW(wide_model.parameters())
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=schedule(
        skip_first=1,
        wait=1,
        warmup=1,
        active=3,
        repeat=2,
    ),
    record_shapes=True,
    on_trace_ready=on_trace_ready,
) as prof:
    for _, batch in zip(trange(14), train_loader):
        input = batch['input_ids'].to(device)

        logits = wide_model(input)[:,:-1,:].contiguous()
        targets = input[:,1:].to(device).contiguous()

        loss = loss_fun(logits.reshape(-1, len(tokenizer)), targets.reshape(-1))

        opt.zero_grad()
        loss.backward()
        opt.step()

        prof.step()

  0%|          | 0/14 [00:00<?, ?it/s]

In [65]:
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=10
))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      72.833ms        36.22%      72.833ms      24.278ms             3  
    autograd::engine::evaluate_function: AddmmBackward0         0.05%     513.747us         0.42%       4.195ms     174.776us       0.000us         0.00%      61.023ms       2.543ms            24  
         

In [73]:
torch.cuda.synchronize()

torch.cuda.memory._record_memory_history(max_entries=100000)
deep_model = Llama(LlamaConfig(n_layers=20, n_heads=4, vocab_size=len(tokenizer), hidden_size=72)).to(device)


opt = torch.optim.AdamW(deep_model.parameters())


for _, batch in zip(trange(10), train_loader):
    input = batch['input_ids'].to(device)

    logits = deep_model(input)[:,:-1,:]
    targets = input[:,1:].to(device)

    loss = loss_fun(logits.reshape(-1, len(tokenizer)), targets.reshape(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()


torch.cuda.memory._dump_snapshot("deep_trace_mem.pickle")


torch.cuda.memory._record_memory_history(enabled=None)

  0%|          | 0/10 [00:00<?, ?it/s]

In [72]:
torch.cuda.synchronize()

torch.cuda.memory._record_memory_history(max_entries=100000)
wide_model = Llama(LlamaConfig(n_layers=1, n_heads=4, vocab_size=len(tokenizer), hidden_size=88)).to(device)

opt = torch.optim.AdamW(wide_model.parameters())


for _, batch in zip(trange(10), train_loader):
    input = batch['input_ids'].to(device)

    logits = wide_model(input)[:,:-1,:].contiguous()
    targets = input[:,1:].to(device).contiguous()

    loss = loss_fun(logits.reshape(-1, len(tokenizer)), targets.reshape(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()


torch.cuda.memory._dump_snapshot("wide_trace_mem.pickle")


torch.cuda.memory._record_memory_history(enabled=None)

  0%|          | 0/10 [00:00<?, ?it/s]

In [46]:
from tqdm import tqdm

def train_epoch(model: Llama, loader, optimizer, loss_fun, writer, print_every, n_epoch, log_prefix='deep_'):
    epoch_loss = 0
    step = 0
    length = len(loader)

    for batch in tqdm(loader):
        input = batch['input_ids'].to(device)

        logits = model(input)[:,:-1,:].contiguous()
        targets = input[:,1:].to(device).contiguous()

        loss = loss_fun(logits.reshape(-1, len(tokenizer)), targets.reshape(-1))

        epoch_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % print_every == 1:
                print(f"[step {step}] train_loss={loss.item():.4f}")
                if writer:
                    writer.add_scalar(f"{log_prefix}loss/train", loss.item(), step + n_epoch*length)

        step += 1

    return epoch_loss // len(loader)


@torch.no_grad()
def val_epoch(model: Llama, loader, loss_fun):
    model.eval()
    epoch_loss = 0
    for batch in tqdm(loader):
        input = batch['input_ids'].to(device)
        logits = model(input)[:,:-1,:].contiguous()
        targets = input[:,1:].to(device).contiguous()

        loss = loss_fun(logits.reshape(-1, len(tokenizer)), targets.reshape(-1))

        epoch_loss += loss.item()

    return epoch_loss // len(loader)

In [47]:
from torch.utils.tensorboard import SummaryWriter
import os
log_dir = "tensorboard"
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)
print(f"[TensorBoard] logging to {log_dir}")

[TensorBoard] logging to tensorboard


In [60]:
deep_model = Llama(LlamaConfig(n_layers=20, n_heads=4, vocab_size=len(tokenizer), hidden_size=72)).to(device)
n_epochs = 1
optimizer = torch.optim.AdamW(deep_model.parameters(), weight_decay=0.1)
ce_loss = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

for epoch in range(n_epochs):
    epoch_loss = train_epoch(deep_model, train_loader, optimizer, ce_loss, writer, print_every=50, n_epoch=epoch)
    val_loss =  val_epoch(deep_model, val_loader, ce_loss)
    if writer:
        writer.add_scalar("[DEEP] epoch_loss/train", epoch_loss, epoch)
        writer.add_scalar("[DEEP] epoch_loss/val", val_loss, epoch)


    print(f"Epoch {epoch}: val_loss={val_loss:.4f}")

  1%|          | 2/316 [00:01<02:42,  1.93it/s]

[step 1] train_loss=10.3228


 16%|█▋        | 52/316 [00:25<02:00,  2.19it/s]

[step 51] train_loss=6.1627


 32%|███▏      | 102/316 [00:48<01:47,  1.99it/s]

...

[step 251] train_loss=4.8593


 96%|█████████▌| 302/316 [02:22<00:07,  1.80it/s]

[step 301] train_loss=4.2959


100%|██████████| 56/56 [00:21<00:00,  2.60it/s]

Epoch 0: val_loss=4.0000


In [61]:
wide_model = Llama(LlamaConfig(n_layers=1, n_heads=4, vocab_size=len(tokenizer), hidden_size=88)).to(device)
optimizer = torch.optim.AdamW(wide_model.parameters())

for epoch in range(n_epochs):
    epoch_loss = train_epoch(wide_model, train_loader, optimizer, ce_loss, writer, print_every=100, n_epoch=epoch, log_prefix="wide_")
    val_loss =  val_epoch(wide_model, val_loader, ce_loss)
    if writer:
        writer.add_scalar("[WIDE] epoch_loss/train", epoch_loss, epoch)
        writer.add_scalar("[WIDE] epoch_loss/val", val_loss, epoch)


    print(f"Epoch {epoch}: val_loss={val_loss:.4f}")

  1%|          | 2/316 [00:00<01:45,  2.98it/s]

[step 1] train_loss=10.4393


 32%|███▏      | 102/316 [00:36<01:46,  2.01it/s]

[step 101] train_loss=5.9670


 64%|██████▍   | 202/316 [01:11<00:43,  2.61it/s]

[step 201] train_loss=5.6111


 96%|█████████▌| 302/316 [01:47<00:06,  2.04it/s]

[step 301] train_loss=5.1497


100%|██████████| 56/56 [00:20<00:00,  2.79it/s]

Epoch 0: val_loss=5.0000


Большая часть параметров это embeddings и последний слой для предсказания токенов. Поэтому при небольшом увеличение dim для глубокой модели, в широкой модели число слоев приходилось увеличивать сильно для сохранения соотоношения параметров. Глубокая модель страдает взрыванием градиентов, пришлось weight_decay побольше поставить. Время обучения от глубины линейно зависит, а от ширины быстрее, так как матричные умножения dim x dim. Но лучше широкая модель, если в память влезает.
